In [1]:
import re
import pandas as pd

In [11]:
# Load the gene effect data
gene_effect = pd.read_csv('../data/ext/CRISPRGeneEffect.csv')# , index_col='ModelID')
gene_effect.rename(columns={'Unnamed: 0':'ModelID'}, inplace=True)
gene_effect.set_index('ModelID', inplace=True)
# Load gene metadata
# gene_meta = pd.read_csv('../data/ext/Gene.csv')

In [15]:
train_df = pd.read_csv('../data/training_data_means.csv')

In [18]:
all_target_names = train_df.columns.tolist()[1:]

In [12]:
# Clean column names: extract just the gene symbol (remove " (ENTREZ_ID)")
def extract_gene_symbol(col_name):
    """
    Extract gene symbol from DepMap format: 'GENE (ID)' → 'GENE'
    """
    # Match everything before the space+parenthesis
    match = re.match(r'^(.+?)\s*\(\d+\)$', col_name)
    if match:
        return match.group(1).strip()
    else:
        # If no pattern match, return as-is (might already be clean)
        return col_name.strip()

# Apply to all columns
gene_effect.columns = [extract_gene_symbol(col) for col in gene_effect.columns]

In [13]:
# Verify the cleaning worked
print(f"Before cleaning (first 5): {gene_effect.columns[:5].tolist()}")

# Check for duplicate gene symbols (can happen with ambiguous mappings)
duplicates = gene_effect.columns[gene_effect.columns.duplicated()].unique()
if len(duplicates) > 0:
    print(f"⚠️  Warning: {len(duplicates)} duplicate gene symbols found")
    print(f"Examples: {duplicates[:10].tolist()}")
    # Option 1: Keep first occurrence
    gene_effect = gene_effect.loc[:, ~gene_effect.columns.duplicated(keep='first')]
    # Option 2: Average duplicates (more conservative)
    # gene_effect = gene_effect.T.groupby(gene_effect.columns).mean().T

Before cleaning (first 5): ['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2']


In [20]:
# After cleaning, check overlap with your target genes
clean_genes = set(gene_effect.columns)
target_genes_set = set(all_target_names)

overlap = clean_genes.intersection(target_genes_set)
print(f"Gene matching after cleaning:")
print(f"  DepMap genes: {len(clean_genes):,}")
print(f"  Target genes: {len(target_genes_set):,}")
print(f"  Overlap: {len(overlap):,} ({len(overlap)/len(target_genes_set)*100:.1f}%)")
print(f"  Missing from DepMap: {len(target_genes_set - overlap):,}")

# Check a few specific genes
for gene in ['TP53', 'MYC', 'ALDOA', 'ACLY'][:4]:
    if gene in clean_genes:
        print(f"  ✓ {gene} found")
    else:
        print(f"  ✗ {gene} NOT found")

Gene matching after cleaning:
  DepMap genes: 18,435
  Target genes: 5,127
  Overlap: 4,967 (96.9%)
  Missing from DepMap: 160
  ✓ TP53 found
  ✓ MYC found
  ✓ ALDOA found
  ✓ ACLY found


In [21]:
def categorize_gene_essentiality(gene_name, gene_effect_df, essential_threshold=-0.5):
    """
    Categorize a gene based on DepMap CRISPR dependency.
    """
    if gene_name not in gene_effect_df.columns:
        return 'non_essential'
    
    # Get effect scores for this gene across all cell lines
    scores = gene_effect_df[gene_name].dropna()
    
    if len(scores) == 0:
        return 'non_essential'
    
    # Fraction of cell lines where gene is essential
    frac_essential = (scores < essential_threshold).mean()
    
    if frac_essential >= 0.8:
        return 'core_essential'
    elif frac_essential >= 0.3:
        return 'contextual_essential'
    else:
        return 'non_essential'

# Clean columns first
## gene_effect.columns = [extract_gene_symbol(col) for col in gene_effect.columns]

# Remove duplicates if any
gene_effect = gene_effect.loc[:, ~gene_effect.columns.duplicated(keep='first')]

# Categorize all target genes
gene_categories = {
    gene: categorize_gene_essentiality(gene, gene_effect)
    for gene in all_target_names
}

In [23]:
print(f"Categorization complete:")
print(f"  core_essential: {sum(v == 'core_essential' for v in gene_categories.values())}")
print(f"  contextual_essential: {sum(v == 'contextual_essential' for v in gene_categories.values())}")
print(f"  non_essential: {sum(v == 'non_essential' for v in gene_categories.values())}")

Categorization complete:
  core_essential: 400
  contextual_essential: 262
  non_essential: 4465


In [24]:
# Save for diagnostic tracking
pd.Series(gene_categories).to_csv('../data/processed/gene_essentiality_categories.csv')

In [28]:
gene_categories = pd.read_csv('../data/processed/gene_essentiality_categories.csv', index_col=0)["0"].to_dict()